### SCD Type 1

Reading the complete Bronze products table

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *
# DBTITLE 1,
df_products = spark.sql('''
                        select * from parquet.`/Volumes/workspace/default/medallion_data/output/bronze/products/`
                        ''')

In [0]:
df_products=df_products.withColumn("category", when(col('category') == 'Elect', 'Electronics').otherwise(col('category'))) \
    .withColumn('price', when(
        col('price').isNull() | col('price').isin('', 'null', "NULL", 'NA', 'N/A', '-'),
        None
    ).otherwise(regexp_replace(col('price'), r"[^0-9.-]", '').cast("double"))) 


Group/Window by prod_id and order by event_ts descending. Keep only the most recent row (row number 1).

In [0]:
df_products=df_products.withColumn("RN",row_number().over(Window.partitionBy("product_id").orderBy(col("ingestion_timestamp").desc()))).filter(col('RN')==1).drop("RN")

unique records to the Silver products table using .mode("overwrite").

In [0]:
df_products.write.mode("overwrite").format("parquet").save("/Volumes/workspace/default/medallion_data/output/silver/products")